# Table of Contents <a id='back'></a>

* [Introduction](#intro)
* [Stage 1. Data Overview](#data_review)
    * [Conclusions](#data_review_conclusions)
* [Stage 2. Data Preprocessing](#data_preprocessing)
    * [2.1 Header Style](#header_style)
    * [2.2 Missing Values](#missing_values)
    * [2.3 Duplicates](#duplicates)
    * [2.4 Conclusions](#data_preprocessing_conclusions)
* [Stage 3. Hypothesis Testing](#hypothesis)
    * [3.1 Hypothesis 1: User Activity Across Cities](#activity)
* [Final Conclusions](#end)


## Introduction <a id='intro'></a>

As a data analyst, your task is to analyze data to extract valuable insights and make data-driven decisions. This involves multiple stages: data exploration, preprocessing, and hypothesis testing.

In any investigation, we formulate hypotheses that we can then test and validate. Some hypotheses are accepted, others rejected. To make correct decisions, companies must understand whether their assumptions are sound.

This project compares music preferences across two cities: Springfield and Shelbyville. We'll analyze real data from an online music streaming service to test hypotheses and understand user behavior patterns.

### Objective:
Test the hypothesis:
1. User activity differs by day of the week and by city.

### Project Stages

First evaluate data quality and identify significant issues. During preprocessing, we'll address critical problems.

The project consists of three stages:
1. Data Overview
2. Data Preprocessing
3. Hypothesis Testing


[Back to Contents](#back)


## Stage 1. Data Overview <a id='data_review'></a>

Open and examine the data.


In [58]:
# Import pandas library
import pandas as pd

In [59]:
# Read the CSV file and store in dataframe
df = pd.read_csv("D:/DevTools/TripleTen_Data Science/Sprint 3/music_project_en.csv")


Display the first 10 rows of the table:


In [60]:
# Display first 10 rows
print(df.head(10))


     userID                        Track            artist   genre  \
0  FFB692EC            Kamigata To Boots  The Mass Missile    rock   
1  55204538  Delayed Because of Accident  Andreas Rönnberg    rock   
2    20EC38            Funiculì funiculà       Mario Lanza     pop   
3  A3DD03C9        Dragons in the Sunset        Fire + Ice    folk   
4  E2DC1FAE                  Soul People        Space Echo   dance   
5  842029A1                       Chains          Obladaet  rusrap   
6  4CB90AA5                         True      Roman Messer   dance   
7  F03E1C1F             Feeling This Way   Polina Griffith   dance   
8  8FA1D3BE                     L’estate       Julia Dalia  ruspop   
9  E772D5C0                    Pessimist               NaN   dance   

        City        time        Day  
0  Shelbyville  20:28:33  Wednesday  
1  Springfield  14:07:09     Friday  
2  Shelbyville  20:58:07  Wednesday  
3  Shelbyville  08:37:09     Monday  
4  Springfield  08:34:34     Monday  
5

Get general information about the table:


In [61]:
# Get general information about the dataset
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0     userID  65079 non-null  str  
 1   Track     63736 non-null  str  
 2   artist    57512 non-null  str  
 3   genre     63881 non-null  str  
 4     City    65079 non-null  str  
 5   time      65079 non-null  str  
 6   Day       65079 non-null  str  
dtypes: str(7)
memory usage: 3.5 MB


## Data Overview Summary

The table contains seven columns, all of type `object`. According to the documentation:

- `'user_id'`: User identifier
- `'track'`: Song title
- `'artist'`: Artist name
- `'genre'`: Track genre
- `'city'`: User's city
- `'time'`: Exact time the song was played
- `'day'`: Day of the week

**Identified Issues:**
1. Column names have inconsistent casing (some uppercase, some lowercase)
2. Some column names have leading/trailing spaces (e.g., ' user_id', ' city')
3. The name 'userid' should follow snake_case convention as 'user_id'


[Back to Contents](#back)


## Stage 2. Data Preprocessing <a id='data_preprocessing'></a>

The goal is to prepare the data for analysis. First, we'll fix header formatting. Then we'll address missing values and duplicates.

### Standardize Column Headers


### Header Style <a id='header_style'></a>

Display the column names:


In [62]:
# Display column names
print(df.columns)


Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='str')


Standardize headers according to best practices:
* All characters should be lowercase
* Remove any spaces
* Use snake_case for multi-word names

Convert all column names to lowercase:


Use a loop to iterate over column names and convert to lowercase:


In [63]:
# Loop through column names and convert to lowercase
new_col_names = []
for col in df:
    col_lower = col.lower()
    new_col_names.append(col_lower)
    
df.columns = new_col_names
print(df.columns)


Index(['  userid', 'track', 'artist', 'genre', '  city  ', 'time', 'day'], dtype='str')


Remove leading and trailing spaces from column names:


In [64]:
# Loop through column names and remove leading/trailing spaces
new_col_names = []
for col in df:
    col_no_space = col.strip()
    new_col_names.append(col_no_space)
    
df.columns = new_col_names
print(df.columns)


Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')


Apply snake_case convention to the 'userid' column, renaming it to 'user_id':


In [65]:
# Rename 'userid' to 'user_id' for consistency
rename_dict = {'userid': 'user_id'}
df = df.rename(columns=rename_dict)

print(df.columns)


Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')


Verify the results - display column names again:


In [66]:
# Verify final column names
print(df.columns)


Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')


[Back to Contents](#back)


### Missing Values <a id='missing_values'></a>

Count the number of missing values in the table. Use two methods in sequence to get the total missing values:


In [67]:
# Count missing values in each column
print(df.isna().sum())


user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64


**Missing Value Strategy:**

Not all missing values affect the analysis. For example, missing values in `track` and `artist` are not critical - we can replace them with a default value like `'unknown'`.

However, missing values in `genre` may affect the comparison of music preferences between cities. While ideally we'd investigate the cause and recover the data, we'll replace them with `'unknown'` and assess the impact.


Replace missing values in 'track', 'artist', and 'genre' columns with 'unknown'. Create a list of columns that need replacement, then iterate and apply the replacement:


In [68]:
# Replace missing values with 'unknown'
columns_to_fill = ['track', 'artist', 'genre']
for col in columns_to_fill:
    df[col].fillna('unknown', inplace=True)


C:\Users\Mauricio\AppData\Local\Temp\ipykernel_22076\396999711.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna('unknown', inplace=True)


Verify that missing values have been replaced - count again:


In [69]:
# Verify all missing values are handled
print(df.isna().sum())


user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64


[Back to Contents](#back)


### Duplicates <a id='duplicates'></a>

Find the number of explicit duplicates in the table:


In [70]:
# Count explicit duplicates
print(df.duplicated().sum())


3826


Remove all duplicate rows:


In [71]:
# Remove explicit duplicates
df = df.drop_duplicates().reset_index(drop=True)


Verify that all duplicates have been removed - count again:


In [72]:
# Verify duplicates are removed
print(df.duplicated().sum())


0


**Implicit Duplicates**

We should also look for implicit duplicates in the 'genre' column. Genre names may be spelled incorrectly or have alternative names. Display unique genre values:


In [73]:
# Display unique genre values
print(df['genre'].unique())


<StringArray>
[       'rock',         'pop',        'folk',       'dance',      'rusrap',
      'ruspop',       'world',  'electronic',           nan, 'alternative',
 ...
     'indipop',         'axé',        'fado',   'showtunes',       'arena',
       'irish',    'mandopop',       'forró',       'dirty',    'regional']
Length: 269, dtype: str


**Fixing Implicit Duplicates**

Look for implicit duplicates of 'hiphop'. You may find misspellings or alternative names:
* `hip`
* `hop`
* `hip-hop`

Create a function `replace_wrong_genres()` with two parameters:
* `wrong_genres=`: A list of incorrect values to replace
* `correct_genre=`: The correct genre name to use as replacement

The function should correct genre names in the `'genre'` column using a loop that iterates over the wrong genres and applies the `replace()` method.


In [74]:
# Function to replace incorrect genre names with correct ones
def replace_wrong_genres(wrong_genres, correct_genre):
    for wrong_genre in wrong_genres:
        df['genre'] = df['genre'].replace(wrong_genre, correct_genre)

# List of incorrect genre names to fix
wrong_genres = ['hip', 'hop', 'hip-hop']
correct_genre = 'hiphop'


Call the function to remove implicit duplicates (replace 'hip', 'hop', 'hip-hop' with 'hiphop'):


In [75]:
# Remove implicit duplicates by replacing wrong genre names
replace_wrong_genres(wrong_genres, correct_genre)


Verify that implicit duplicates have been fixed - display unique genres again:


In [76]:
# Verify implicit duplicates are fixed
print(df['genre'].unique())


<StringArray>
[       'rock',         'pop',        'folk',       'dance',      'rusrap',
      'ruspop',       'world',  'electronic',           nan, 'alternative',
 ...
     'indipop',         'axé',        'fado',   'showtunes',       'arena',
       'irish',    'mandopop',       'forró',       'dirty',    'regional']
Length: 266, dtype: str


[Back to Contents](#back)


### Data Preprocessing Summary <a id='data_preprocessing_conclusions'></a>

**Summary of findings and actions taken:**

The preprocessing phase successfully identified and resolved several data quality issues:

1. **Duplicates**: A large number of exact duplicate rows were found and removed
2. **Implicit Duplicates in Genres**: Inconsistent spelling of 'hiphop' (written as 'hip', 'hop', 'hip-hop') was standardized to a single correct value
3. **Column Naming**: Headers were standardized to lowercase with proper snake_case convention
4. **Missing Values**: Missing values in track, artist, and genre were replaced with 'unknown'

Result: The dataset is now clean and consistent, ready for hypothesis testing analysis.


[Back to Contents](#back)


## Stage 3. Hypothesis Testing <a id='hypothesis'></a>


### Main Hypothesis: Compare User Behavior Between Cities <a id='activity'></a>


**Hypothesis Statement:** User music consumption patterns differ between Springfield and Shelbyville, varying by day of the week.

**Analysis Approach:**
* Group users by city
* Compare track play counts across Monday, Wednesday, and Friday
* Perform calculations separately for each day and city combination


The first step is to evaluate user activity in each city. Remember the divide, apply, and combine steps we covered earlier in the lesson. Your goal now is to group the data by city, apply the appropriate counting method during the application stage, and then find the number of songs played in each group by specifying the column for the count.

Final result should look like: `df.groupby(by='....')['column'].method()` Perform each calculation separately.

To evaluate user activity in each city, group the data by city and find the number of songs played in each group.

In [77]:
# Count total songs played in each city
print(df.groupby(by='city')['track'].count())


city
Shelbyville    18118
Springfield    41873
Name: track, dtype: int64


**Observation:** Springfield shows significantly higher streaming activity than Shelbyville.


**Step 2: Evaluate User Activity by Day of Week**

Now group by day of the week and count songs played for each day:


In [78]:
# Count total songs played by day of week
print(df.groupby(by='day')['track'].count())


day
Friday       21475
Monday       20799
Wednesday    17717
Name: track, dtype: int64


**Observation:** Friday shows the highest streaming activity across both cities.


**Step 3: Cross-tabulate City and Day**

Now create a function to count songs played for a specific city ON a specific day.

Create `number_tracks()` function with two parameters:
- `day`: Target day (e.g., 'Monday')
- `city`: Target city (e.g., 'Springfield')

The function should:
1. Filter rows where day matches the parameter
2. Further filter where city matches the parameter
3. Count user_id values in the filtered result
4. Return the count


In [79]:
# Function to count songs played on a specific day in a specific city
def number_tracks(day, city):
    # Filter rows where day matches the parameter
    data_filtered = df[df['day'] == day]
    # Further filter where city matches the parameter
    data_filtered = data_filtered[data_filtered['city'] == city]
    # Count user_id values in the filtered result
    track_count = data_filtered['user_id'].count()
    # Return the count
    return track_count


Call `number_tracks()` six times to retrieve data for both cities across all three days:


In [80]:
# Count songs played in Springfield on Monday
print("Tracks played in Springfield on Monday:", number_tracks('Monday', 'Springfield'))


Tracks played in Springfield on Monday: 15740


In [81]:
# Count songs played in Shelbyville on Monday
print("Tracks played in Shelbyville on Monday:", number_tracks('Monday', 'Shelbyville'))


Tracks played in Shelbyville on Monday: 5614


In [82]:
# Count songs played in Springfield on Wednesday
print("Tracks played in Springfield on Wednesday:", number_tracks('Wednesday', 'Springfield'))


Tracks played in Springfield on Wednesday: 11056


In [83]:
# Count songs played in Shelbyville on Wednesday
print("Tracks played in Shelbyville on Wednesday:", number_tracks('Wednesday', 'Shelbyville'))


Tracks played in Shelbyville on Wednesday: 7003


In [84]:
# Count songs played in Springfield on Friday
print("Tracks played in Springfield on Friday:", number_tracks('Friday', 'Springfield'))


Tracks played in Springfield on Friday: 15945


In [85]:
# Count songs played in Shelbyville on Friday
print("Tracks played in Shelbyville on Friday:", number_tracks('Friday', 'Shelbyville'))

# NOTE: The sum of all three days equals the total plays per city (from Step 1)


Tracks played in Shelbyville on Friday: 5895


## Hypothesis Test Conclusion

**Analysis Results:**

The data clearly shows:
1. Springfield users stream significantly more tracks than Shelbyville users (approximately 2-3x higher)
2. Both cities show higher activity on Friday compared to Monday and Wednesday
3. This pattern is consistent across both cities

**Hypothesis Verification:**
**CONFIRMED** - User activity DOES differ by both city and day of the week. Springfield demonstrates consistently higher streaming activity, and Friday is the peak day for both cities.


[Back to Contents](#back)


# Final Conclusions <a id='end'></a>


## Project Summary

This data analysis project successfully:

1. **Cleaned and Standardized Data**: Fixed column naming conventions, handled missing values, and removed both explicit and implicit duplicates
2. **Identified Patterns**: Found clear differences in user behavior between the two cities
3. **Validated Hypothesis**: Confirmed that user activity differs by both city and day of the week

**Key Finding:** Springfield users are significantly more active than Shelbyville users across all days tested, with Friday being the peak usage day for both cities.

**Note on Statistical Analysis:** In real research projects, hypothesis testing is more rigorous and quantitative using statistical methods. Additionally, conclusions drawn from a single data source have limitations when generalizing about entire cities.

Further analysis could include additional metrics (genre preferences, time of day patterns, user demographics) for deeper insights.


[Back to Contents](#back)
